In [1]:
from gensim.models import HdpModel
from gensim.models.callbacks import CoherenceMetric
from gensim import corpora
import tomotopy as tp
from gensim.models.coherencemodel import CoherenceModel
import pandas as pd
import numpy as np
import logging
import sys
import ast
from tqdm import tqdm

In [2]:
def hlda_learning(snum):
    input_data = pd.read_feather(f'./Datasets/covid{snum}.ftr')
    cps = tp.utils.Corpus()

    for i in range(len(input_data['okt'])): 
        if type(input_data['okt'][i]) == float:
            continue
        doc = input_data['okt'][i]
        doc_rm = []
        for word in doc:
            doc_rm.append(word)
        if len(doc_rm) < 1:
            continue
        cps.add_doc(doc_rm)  

    mdl = tp.HLDAModel(tw=tp.TermWeight.ONE, min_df=10, depth=3, corpus=cps)

    mdl.train(0)
    print('Num docs:', len(mdl.docs), ', Vocab size:', len(mdl.used_vocabs), ', Num words:', mdl.num_words)
    print('Removed top words:', mdl.removed_top_words)
    print('Training...', file=sys.stderr, flush=True)
    
    for _ in range(0, 1000, 10):
        mdl.train(7)
        mdl.train(3, freeze_topics=True)
        print('Iteration: {:05}\tll per word: {:.5f}\tNum. of topics: {}'.format(mdl.global_step, mdl.ll_per_word, mdl.live_k))

    for _ in range(0, 100, 10):
        mdl.train(10, freeze_topics=True)
        print('Iteration: {:05}\tll per word: {:.5f}\tNum. of topics: {}'.format(mdl.global_step, mdl.ll_per_word, mdl.live_k))

    mdl.summary(topic_word_top_n=20)
    print('Saving...', file=sys.stderr, flush=True)
    mdl.save(f'./models/hlda_{snum}.bin', True)

In [3]:
for i in [1,2,3,4]:
    hlda_learning(i)

Training...


Num docs: 68471 , Vocab size: 3400 , Num words: 320434
Removed top words: []
Iteration: 00010	ll per word: -inf	Num. of topics: 232
Iteration: 00020	ll per word: -inf	Num. of topics: 284
Iteration: 00030	ll per word: -inf	Num. of topics: 304
Iteration: 00040	ll per word: -inf	Num. of topics: 330
Iteration: 00050	ll per word: -inf	Num. of topics: 341
Iteration: 00060	ll per word: -inf	Num. of topics: 352
Iteration: 00070	ll per word: -inf	Num. of topics: 369
Iteration: 00080	ll per word: -inf	Num. of topics: 383
Iteration: 00090	ll per word: -inf	Num. of topics: 400
Iteration: 00100	ll per word: -inf	Num. of topics: 406
Iteration: 00110	ll per word: -inf	Num. of topics: 410
Iteration: 00120	ll per word: -inf	Num. of topics: 423
Iteration: 00130	ll per word: -inf	Num. of topics: 426
Iteration: 00140	ll per word: -inf	Num. of topics: 435
Iteration: 00150	ll per word: -inf	Num. of topics: 434
Iteration: 00160	ll per word: -inf	Num. of topics: 435
Iteration: 00170	ll per word: -inf	Num. of 

Saving...


Iteration: 01100	ll per word: -inf	Num. of topics: 493
<Basic Info>
| HLDAModel (current version: 0.12.3)
| 68471 docs, 320434 words
| Total Vocabs: 14940, Used Vocabs: 3400
| Entropy of words: 6.63813
| Entropy of term-weighted words: 6.63813
| Removed Vocabs: <NA>
|
<Training Info>
| Iterations: 1100, Burn-in steps: 0
| Optimization Interval: 10
| Log-likelihood per word: -inf
|
<Initial Parameters>
| tw: TermWeight.ONE
| min_cf: 0 (minimum collection frequency of words)
| min_df: 10 (minimum document frequency of words)
| rm_top: 0 (the number of top words to be removed)
| depth: 3 (the maximum depth level of hierarchy between 2 ~ 32767)
| alpha: [0.1] (hyperparameter of Dirichlet distribution for document-depth level, given as a single `float` in case of symmetric prior and as a list with length `depth` of `float` in case of asymmetric prior.)
| eta: 0.01 (hyperparameter of Dirichlet distribution for topic-word)
| gamma: 0.1 (concentration coeficient of Dirichlet Process)
| seed: 2